In [ ]:
import os
import dotenv
from logfire.query_client import LogfireQueryClient

# Load credentials
dotenv.load_dotenv("/workspaces/.codespaces/shared/.env")

# Create Logfire query client
read_token = os.getenv("LOGFIRE_READ_TOKEN")
logfire_query_client = LogfireQueryClient(
    read_token=read_token
)

print("Logfire Query Client initialized")

In [ ]:
from datetime import datetime, timedelta, timezone

trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'agent run'
       OR span_name LIKE 'invoke_agent %'
    ORDER BY start_timestamp DESC
    LIMIT 5
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

trace_rows["rows"]

In [ ]:
trace_id = trace_rows["rows"][0]["trace_id"]

run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' AS all_messages,
        attributes->>'gen_ai.usage.input_tokens' AS input_tokens,
        attributes->>'gen_ai.usage.output_tokens' AS output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND (
          span_name = 'agent run'
          OR span_name LIKE 'invoke_agent %'
      )
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

print("Agent runs:", len(run_row["rows"]))
print("Input tokens:", [
    row["input_tokens"] for row in run_row["rows"]
])
print("Output tokens:", [
    row["output_tokens"] for row in run_row["rows"]
])
print("Messages per run:", [
    len(row["all_messages"] or []) for row in run_row["rows"]
])

In [ ]:
attributes_result = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT attributes
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    LIMIT 1
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

if attributes_result["rows"]:
    print(list(attributes_result["rows"][0]["attributes"].keys()))
else:
    print("No matching records")

In [ ]:
usage_result = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        start_timestamp,
        attributes->>'gen_ai.aggregated_usage.input_tokens'
            AS input_tokens,
        attributes->>'gen_ai.aggregated_usage.output_tokens'
            AS output_tokens,
        attributes->>'gen_ai.aggregated_usage.cache_read.input_tokens'
            AS cached_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

usage_result["rows"]

In [ ]:
evaluation_rows = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        start_timestamp,
        attributes->'final_result' AS final_result
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]
    print(f"Run {i}:")
    print("Result type:", type(result).__name__)
    print("Fields:", list(result.keys()) if isinstance(result, dict) else "Not a dictionary")
    print()

In [ ]:
evaluation_data = []

for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]

    evaluation_data.append({
        "run": i,
        "found_answer": result.get("found_answer"),
        "confidence": result.get("confidence"),
        "answer_type": result.get("answer_type"),
        "references_count": len(result.get("references") or []),
        "answer_length": len(result.get("answer") or ""),
    })

evaluation_data

In [ ]:
for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]
    print(f"Run {i}: {result.get('answer', '')[:150]!r}")

In [ ]:
scrub_result = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        start_timestamp,
        attributes->'logfire.scrubbed' AS scrubbed
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'invoke_agent search'
    ORDER BY start_timestamp ASC
    """,
    min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
)

scrub_result["rows"]

In [ ]:
evaluation_dataset = []

for i, row in enumerate(evaluation_rows["rows"], start=1):
    result = row["final_result"]
    answer = result.get("answer", "")

    is_scrubbed = (
        answer.startswith("[Scrubbed due to")
    )

    evaluation_dataset.append({
        "run": i,
        "answer": None if is_scrubbed else answer,
        "found_answer": result.get("found_answer"),
        "confidence": result.get("confidence"),
        "references_count": len(result.get("references") or []),
        "scrubbed": is_scrubbed,
        "eligible_for_text_evaluation": not is_scrubbed,
    })

for item in evaluation_dataset:
    print({
        key: value
        for key, value in item.items()
        if key != "answer"
    })

In [14]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

run = run_row["rows"][0]

messages = ModelMessagesTypeAdapter.validate_python(
    run["all_messages"]
)

print("Recovered messages:", len(messages))
print("First message type:", type(messages[0]).__name__)

ValidationError: 7 validation errors for list[tagged-union[ModelRequest,ModelResponse]]
0
  Unable to extract tag using discriminator 'kind' [type=union_tag_not_found, input_value={'role': 'user', 'parts':...t is LLM as a Judge?'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_not_found
1
  Unable to extract tag using discriminator 'kind' [type=union_tag_not_found, input_value={'role': 'assistant', 'pa...'finish_reason': 'stop'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_not_found
2
  Unable to extract tag using discriminator 'kind' [type=union_tag_not_found, input_value={'role': 'user', 'parts':...quickstart_llm.mdx'}]}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_not_found
3
  Unable to extract tag using discriminator 'kind' [type=union_tag_not_found, input_value={'role': 'assistant', 'pa...'finish_reason': 'stop'}, input_type

In [15]:
messages = run["all_messages"]

print("Container:", type(messages).__name__)
print("First message keys:", list(messages[0].keys()))
print("First message role:", messages[0].get("role"))
print("First part keys:", list(messages[0]["parts"][0].keys()))

Container: list
First message keys: ['role', 'parts']
First message role: user
First part keys: ['type', 'content']


In [16]:
for i, msg in enumerate(messages):
    print(
        i,
        msg.get("role"),
        [part.get("type") for part in msg.get("parts", [])]
    )

0 user ['text']
1 assistant ['tool_call']
2 user ['tool_call_response']
3 assistant ['tool_call', 'tool_call']
4 user ['tool_call_response', 'tool_call_response']
5 assistant ['tool_call']
6 user ['tool_call_response']


In [17]:
def reconstruct_trace(messages):
    events = []

    for message in messages:
        role = message.get("role")

        for part in message.get("parts", []):
            events.append({
                "role": role,
                "event_type": part.get("type"),
                "has_content": "content" in part,
            })

    return events


reconstructed_trace = reconstruct_trace(messages)

for event in reconstructed_trace:
    print(event)
    

{'role': 'user', 'event_type': 'text', 'has_content': True}
{'role': 'assistant', 'event_type': 'tool_call', 'has_content': False}
{'role': 'user', 'event_type': 'tool_call_response', 'has_content': False}
{'role': 'assistant', 'event_type': 'tool_call', 'has_content': False}
{'role': 'assistant', 'event_type': 'tool_call', 'has_content': False}
{'role': 'user', 'event_type': 'tool_call_response', 'has_content': False}
{'role': 'user', 'event_type': 'tool_call_response', 'has_content': False}
{'role': 'assistant', 'event_type': 'tool_call', 'has_content': False}
{'role': 'user', 'event_type': 'tool_call_response', 'has_content': False}


In [18]:
original_prompt = next(
    part["content"]
    for msg in messages
    if msg.get("role") == "user"
    for part in msg.get("parts", [])
    if part.get("type") == "text"
)

print("Original question:", original_prompt)

Original question: What is LLM as a Judge?


In [19]:
import sys
import asyncio

sys.path.insert(0, "/workspaces/ai-engineering-buildcamp/04-documentation-agent")

from doc_agent import (
    DocumentationAgentConfig,
    SIMPLE_INSTRUCTIONS,
    create_agent,
)
from tools import create_documentation_tools_cached

tools = create_documentation_tools_cached()
config = DocumentationAgentConfig(instructions=SIMPLE_INSTRUCTIONS)
replay_agent = create_agent(config, tools)

replay_result = await replay_agent.run(original_prompt)

print("Replay completed")
print("Result type:", type(replay_result.output).__name__)

Cache not found, creating documentation tools...
Documentation tools cached to .cache/documentation_tools.pkl.
Replay completed
Result type: str


In [20]:
original_answer = evaluation_rows["rows"][0]["final_result"]["answer"]
replayed_answer = replay_result.output

print("Original answer length:", len(original_answer))
print("Replay answer length:", len(replayed_answer))
print("Exact match:", original_answer == replayed_answer)
print("Original found answer:", evaluation_rows["rows"][0]["final_result"]["found_answer"])

Original answer length: 2168
Replay answer length: 1967
Exact match: False
Original found answer: True


In [22]:
import sys

sys.path.insert(
    0,
    "/workspaces/ai-engineering-buildcamp/04-documentation-agent"
)

import trace_replay

trace = trace_replay.fetch_trace(
    trace_id,
    logfire_query_client
)

run = trace_replay.trace_to_run_result(trace)

print("Result type:", type(run).__name__)
print("Output type:", type(run.output).__name__)
print("Messages:", len(run.all_messages()))
print("Usage:", run.usage())

TypeError: LogfireQueryClient.query_json_rows() missing 1 required keyword-only argument: 'min_timestamp'

In [24]:
import importlib
import trace_replay
import trace_replay.converter

importlib.reload(trace_replay.converter)
importlib.reload(trace_replay)

trace = trace_replay.fetch_trace(
    trace_id,
    logfire_query_client
)

run = trace_replay.trace_to_run_result(trace)

print("Result type:", type(run).__name__)
print("Output type:", type(run.output).__name__)
print("Messages:", len(run.all_messages()))
print("Usage:", run.usage)

Result type: AgentRunResult
Output type: dict
Messages: 19
Usage: RunUsage(input_tokens=110353, output_tokens=2680)


In [25]:
for i, message in enumerate(run.all_messages(), start=1):
    print(
        i,
        type(message).__name__,
        [type(part).__name__ for part in message.parts]
    )

1 ModelRequest ['UserPromptPart']
2 ModelResponse ['ToolCallPart']
3 ModelRequest ['ToolReturnPart']
4 ModelResponse ['ToolCallPart', 'ToolCallPart']
5 ModelRequest ['ToolReturnPart', 'ToolReturnPart']
6 ModelResponse ['ToolCallPart']
7 ModelRequest ['ToolReturnPart', 'UserPromptPart']
8 ModelResponse ['ToolCallPart']
9 ModelRequest ['ToolReturnPart']
10 ModelResponse ['ToolCallPart']
11 ModelRequest ['ToolReturnPart']
12 ModelResponse ['ToolCallPart']
13 ModelRequest ['ToolReturnPart', 'ToolReturnPart', 'UserPromptPart', 'UserPromptPart']
14 ModelResponse ['ToolCallPart']
15 ModelRequest ['ToolReturnPart']
16 ModelResponse ['ToolCallPart', 'ToolCallPart']
17 ModelRequest ['ToolReturnPart', 'ToolReturnPart']
18 ModelResponse ['ToolCallPart']
19 ModelRequest ['ToolReturnPart']


In [26]:
from collections import Counter
from pydantic_ai.messages import ToolCallPart, ToolReturnPart

calls = Counter()
returns = Counter()

for message in run.all_messages():
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            calls[part.tool_call_id] += 1
        elif isinstance(part, ToolReturnPart):
            returns[part.tool_call_id] += 1

print("Tool calls:", sum(calls.values()))
print("Tool returns:", sum(returns.values()))
print("Missing returns:", list((calls - returns).elements()))
print("Unexpected returns:", list((returns - calls).elements()))

Tool calls: 11
Tool returns: 12
Missing returns: []
Unexpected returns: ['call_L7ZweseWTiXVkhff6WCGvG6Y']


In [27]:
for i, row in enumerate(run_row["rows"], start=1):
    messages_in_run = row.get("all_messages") or []

    if isinstance(messages_in_run, str):
        messages_in_run = json.loads(messages_in_run)

    print(
        f"Run {i}:",
        len(messages_in_run),
        "OTel messages"
    )

Run 1: 7 OTel messages
Run 2: 14 OTel messages
Run 3: 20 OTel messages


In [28]:
snapshots = []

for row in run_row["rows"]:
    data = row["all_messages"]
    if isinstance(data, str):
        data = json.loads(data)
    snapshots.append(data)

for i in range(len(snapshots) - 1):
    earlier = snapshots[i]
    later = snapshots[i + 1]

    print(
        f"Run {i + 1} → Run {i + 2}:",
        "Prefix match:",
        earlier == later[:len(earlier)]
    )

Run 1 → Run 2: Prefix match: True
Run 2 → Run 3: Prefix match: False


In [29]:
earlier = snapshots[1]
later = snapshots[2]

for i, (a, b) in enumerate(zip(earlier, later)):
    if a != b:
        print("First difference at index:", i)
        print("Earlier role:", a.get("role"))
        print("Later role:", b.get("role"))
        print(
            "Earlier types:",
            [p.get("type") for p in a.get("parts", [])]
        )
        print(
            "Later types:",
            [p.get("type") for p in b.get("parts", [])]
        )
        break
else:
    print("All compared messages match")
    

First difference at index: 6
Earlier role: user
Later role: user
Earlier types: ['tool_call_response']
Later types: ['tool_call_response', 'text']


In [30]:
for label, snapshot in [
    ("Run 2", snapshots[1]),
    ("Run 3", snapshots[2]),
]:
    msg = snapshot[6]

    print(label)
    for part in msg["parts"]:
        print({
            "type": part.get("type"),
            "id": part.get("id"),
            "name": part.get("name"),
        })

Run 2
{'type': 'tool_call_response', 'id': 'call_L7ZweseWTiXVkhff6WCGvG6Y', 'name': 'final_result'}
Run 3
{'type': 'tool_call_response', 'id': 'call_L7ZweseWTiXVkhff6WCGvG6Y', 'name': 'final_result'}
{'type': 'text', 'id': None, 'name': None}


In [31]:
last_messages = trace_replay.otel_to_model_messages(
    snapshots[-1]
)

last_calls = Counter()
last_returns = Counter()

for message in last_messages:
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            last_calls[part.tool_call_id] += 1
        elif isinstance(part, ToolReturnPart):
            last_returns[part.tool_call_id] += 1

print("Messages:", len(last_messages))
print("Tool calls:", sum(last_calls.values()))
print("Tool returns:", sum(last_returns.values()))
print("Missing returns:", list((last_calls - last_returns).elements()))
print("Unexpected returns:", list((last_returns - last_calls).elements()))

Messages: 19
Tool calls: 11
Tool returns: 11
Missing returns: []
Unexpected returns: []


In [32]:
import importlib
import trace_replay
import trace_replay.converter

importlib.reload(trace_replay.converter)
importlib.reload(trace_replay)

trace = trace_replay.fetch_trace(
    trace_id,
    logfire_query_client
)

run = trace_replay.trace_to_run_result(trace)

print("Messages:", len(run.all_messages()))
print("Input tokens:", run.usage.input_tokens)
print("Output tokens:", run.usage.output_tokens)
print(
    "Matches latest snapshot:",
    len(trace.all_messages) == len(snapshots[-1])
)

Messages: 19
Input tokens: 110353
Output tokens: 2680
Matches latest snapshot: True


In [34]:
trace_ids = list(dict.fromkeys(
    row["trace_id"]
    for row in trace_rows["rows"]
))

traces = trace_replay.fetch_traces(
    trace_ids,
    logfire_query_client
)

runs = [
    trace_replay.trace_to_run_result(trace)
    for trace in traces.values()
]

print("Trace IDs:", len(trace_ids))
print("Recovered runs:", len(runs))

Trace IDs: 2
Recovered runs: 2


In [35]:
import pickle
from pathlib import Path

output_path = Path("data/logs.bin")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("wb") as f:
    pickle.dump(runs, f)

print("Saved runs:", len(runs))
print("File:", output_path)

Saved runs: 2
File: data/logs.bin


In [36]:
with output_path.open("rb") as f:
    loaded_runs = pickle.load(f)

print("Loaded runs:", len(loaded_runs))
print("Result types:", [
    type(run).__name__ for run in loaded_runs
])

Loaded runs: 2
Result types: ['AgentRunResult', 'AgentRunResult']
